# Sparkle V2 — Fase 3.6a: predicción selectiva (curva riesgo-cobertura)

**Protocolo pre-registrado:** `Protocolo_Fase36_Camino_al_70.md` §2 (2026-07-29).

El producto no está obligado a opinar sobre toda ventana: puede **abstenerse** donde la
probabilidad está cerca del umbral. Se reporta la curva riesgo-cobertura completa.

**Por qué esto NO es la trampa de las ventanas homogéneas** (§9.5 del registro): aquellas se
seleccionaban por la **etiqueta**, que en inferencia no existe. La abstención se selecciona por
la **confianza del modelo**, disponible en tiempo real. Es un mecanismo desplegable.

**Congelado antes de correr:** confianza = `|p − 0,50|` · umbral 0,50 sin cambios · trivial
recalculado en cada cobertura · grilla c ∈ {1,00 … 0,50} paso 0,05 · **se elige la MAYOR c**
que cumple accuracy ≥ 0,70 **y** brecha ≥ +0,093 **y** c ≥ 0,60.

Corre sobre las predicciones OOF que ya existen. No reentrena nada. **El Test no se toca.**

In [ ]:
import sys, os, json
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'
else:
    PROJECT_DIR = os.environ.get('SPARKLE_PROJECT_DIR', './SparkleV2')

OUT_DIR = f'{PROJECT_DIR}/fase3_5_proyeccion'
OOF_NPZ = f'{OUT_DIR}/oof_predicciones.npz'
OUT_36  = f'{PROJECT_DIR}/fase3_6_camino70'
os.makedirs(OUT_36, exist_ok=True)

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
import matplotlib.pyplot as plt

# ------- Parámetros CONGELADOS por el pre-registro (§2) -------
UMBRAL       = 0.50
COBERTURAS   = np.round(np.arange(1.00, 0.499, -0.05), 2)
ACC_OBJETIVO = 0.70
BRECHA_MIN   = 0.093          # la del modelo A actual sobre Test
COB_MIN      = 0.60
N_BOOT       = 5000
SEED         = 42
BRAZO        = 'R1_d16'       # congelado en §11.7 del registro

z = np.load(OOF_NPZ, allow_pickle=True)
y = z['y']; part = np.array([str(p) for p in z['part']])
p_main = z[BRAZO]
print(f'OOF: {len(y)} clips · {len(set(part))} participantes · brazo {BRAZO}')
print(f'Objetivo: accuracy ≥ {ACC_OBJETIVO} · brecha ≥ +{BRECHA_MIN} · cobertura ≥ {COB_MIN}')

## 1. Curva riesgo-cobertura

En cada cobertura se reportan **juntas** accuracy, balanced accuracy, F1 macro, el **trivial
recalculado sobre el subconjunto cubierto** y la brecha. La accuracy sola no significa nada
acá: al abstenerse, el trivial también se mueve.

In [ ]:
def curva(p, y=y, coberturas=COBERTURAS, umbral=UMBRAL):
    conf = np.abs(p - umbral)
    orden = np.argsort(-conf)              # de más confiado a menos
    filas = []
    for c in coberturas:
        k = max(1, int(round(c * len(p))))
        sel = orden[:k]
        ys, ps = y[sel], p[sel]
        pred = (ps > umbral).astype(int)
        triv = max((ys == 0).mean(), (ys == 1).mean())
        acc = accuracy_score(ys, pred)
        filas.append({
            'cobertura': float(c), 'n': int(k),
            'accuracy': round(float(acc), 4),
            'trivial': round(float(triv), 4),
            'brecha': round(float(acc - triv), 4),
            'balanced_acc': round(float(balanced_accuracy_score(ys, pred)), 4),
            'f1_macro': round(float(f1_score(ys, pred, average='macro')), 4),
            'pct_desatencion': round(float(ys.mean()), 4),
            'umbral_conf': round(float(conf[orden[k-1]]), 4)})
    return pd.DataFrame(filas)

df = curva(p_main)
print(df.to_string(index=False))
df.to_csv(f'{OUT_36}/curva_riesgo_cobertura_{BRAZO}.csv', index=False)

## 2. Control 1 — abstención aleatoria a la misma cobertura

Si abstenerse al azar sube la accuracy igual, entonces la confianza no informa nada y lo que
se está midiendo es ruido de tamaño de muestra.

In [ ]:
rng = np.random.default_rng(SEED)
filas = []
for c in COBERTURAS:
    k = max(1, int(round(c * len(y))))
    accs, brechas = [], []
    for _ in range(200):
        sel = rng.choice(len(y), size=k, replace=False)
        ys = y[sel]; pred = (p_main[sel] > UMBRAL).astype(int)
        a = accuracy_score(ys, pred); t = max((ys == 0).mean(), (ys == 1).mean())
        accs.append(a); brechas.append(a - t)
    filas.append({'cobertura': float(c),
                  'accuracy_aleatoria': round(float(np.mean(accs)), 4),
                  'brecha_aleatoria': round(float(np.mean(brechas)), 4)})
df_rand = pd.DataFrame(filas)
comp = df[['cobertura', 'accuracy', 'brecha']].merge(df_rand, on='cobertura')
comp['ganancia_acc'] = (comp.accuracy - comp.accuracy_aleatoria).round(4)
comp['ganancia_brecha'] = (comp.brecha - comp.brecha_aleatoria).round(4)
print(comp.to_string(index=False))
if comp.ganancia_acc.max() < 0.01:
    print('\n⚠️  La abstención por confianza no supera a la aleatoria: la confianza NO informa. '
          'El paso 1 no aporta nada real.')

## 3. Control 2 — la misma curva sin la rama de apariencia (R0)

Requiere haber corrido 3.5c, que guardó R0. Si no está, se saltea y se anota.

In [ ]:
R0_CSV = f'{OUT_DIR}/controles_tabla.csv'
p_R0 = z['R0'] if 'R0' in z.files else None
if p_R0 is None:
    print('R0 no está dentro de oof_predicciones.npz.')
    print('Para tenerlo: volver a correr la sección 4 de 3.5c y agregar al final')
    print("    np.savez(OOF_NPZ, **{k: z[k] for k in z.files}, R0=p_R0)")
    print('Se continúa sin este control; queda anotado como pendiente.')
else:
    df0 = curva(p_R0)
    print(df0.to_string(index=False))
    df0.to_csv(f'{OUT_36}/curva_riesgo_cobertura_R0.csv', index=False)
    print('\nSi R0 alcanza el objetivo tan bien como el brazo con apariencia, la selectividad '
          'no depende de la rama de apariencia y eso hay que decirlo.')

## 4. Composición del subconjunto cubierto

Si la abstención es asimétrica, el subconjunto cubierto se desbalancea y el trivial sube por
esa vía y no por mérito del modelo. Es el chequeo que separa un resultado real de un artefacto.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(df.cobertura, df.accuracy, 'o-', label='accuracy')
ax[0].plot(df.cobertura, df.trivial, 's--', label='trivial del cubierto')
ax[0].plot(df.cobertura, df.balanced_acc, '^-', label='balanced acc')
ax[0].axhline(ACC_OBJETIVO, color='r', ls=':', label='objetivo 0,70')
ax[0].axvline(COB_MIN, color='gray', ls=':', label='cobertura mínima')
ax[0].set_xlabel('cobertura'); ax[0].legend(); ax[0].invert_xaxis()
ax[0].set_title('Accuracy vs. trivial: ¿se despegan?')
ax[1].plot(df.cobertura, df.brecha, 'o-', color='tab:green', label='brecha')
ax[1].axhline(BRECHA_MIN, color='r', ls=':', label=f'brecha mínima +{BRECHA_MIN}')
ax[1].axhline(0, color='k', lw=0.8)
ax[1].set_xlabel('cobertura'); ax[1].legend(); ax[1].invert_xaxis()
ax[1].set_title('Brecha vs. trivial — el número que mira el jurado')
plt.tight_layout(); plt.savefig(f'{OUT_36}/curva_riesgo_cobertura.png', dpi=120); plt.show()

print('\n% de Desatención en el cubierto (en el total OOF es '
      f'{y.mean():.4f}):')
print(df[['cobertura', 'pct_desatencion', 'trivial']].to_string(index=False))

## 5. IC bootstrap clusterizado en las coberturas candidatas

In [ ]:
def boot_cobertura(p, c, y=y, part=part, n_boot=N_BOOT, seed=SEED, umbral=UMBRAL):
    '''IC de accuracy y de brecha a cobertura c, remuestreando PARTICIPANTES.
    El umbral de confianza se recalcula DENTRO de cada remuestra: es parte del procedimiento.'''
    rng = np.random.default_rng(seed)
    parts = np.unique(part)
    idx = {q: np.where(part == q)[0] for q in parts}
    conf = np.abs(p - umbral)
    accs, brs = np.empty(n_boot), np.empty(n_boot)
    for i in range(n_boot):
        sel = rng.choice(parts, size=len(parts), replace=True)
        j = np.concatenate([idx[q] for q in sel])
        k = max(1, int(round(c * len(j))))
        jj = j[np.argsort(-conf[j])[:k]]
        ys = y[jj]; pred = (p[jj] > umbral).astype(int)
        a = (ys == pred).mean(); t = max((ys == 0).mean(), (ys == 1).mean())
        accs[i] = a; brs[i] = a - t
    pct = lambda v: [round(float(x), 4) for x in np.percentile(v, [2.5, 97.5])]
    return {'accuracy_IC95': pct(accs), 'brecha_IC95': pct(brs)}

cands = df[(df.accuracy >= ACC_OBJETIVO) & (df.brecha >= BRECHA_MIN) &
           (df.cobertura >= COB_MIN)]
print('Coberturas que cumplen las TRES condiciones del protocolo:')
print(cands.to_string(index=False) if len(cands) else '  (ninguna)')

for c in (sorted(cands.cobertura, reverse=True)[:1] or [0.60, 0.80]):
    r = boot_cobertura(p_main, c)
    print(f'\ncobertura {c:.2f} → accuracy IC95 {r["accuracy_IC95"]} · '
          f'brecha IC95 {r["brecha_IC95"]}')

## 6. Veredicto contra la regla pre-registrada

**Se elige la MAYOR cobertura que cumple las tres condiciones**, no la que da el mejor número.
Maximizar la exactitud eligiendo la cobertura más chica es exactamente el sesgo a evitar.

In [ ]:
print('================ VEREDICTO PASO 1 ================')
if len(cands) == 0:
    mejor = df.loc[df.accuracy.idxmax()]
    print('❌ Ninguna cobertura ≥ 0,60 alcanza accuracy ≥ 0,70 con brecha ≥ +0,093.')
    print(f'   Mejor accuracy observada: {mejor.accuracy:.4f} a cobertura {mejor.cobertura:.2f} '
          f'(brecha {mejor.brecha:+.4f}).')
    print('   Conclusión pre-registrada: el paso 1 se declara FALLIDO y se reporta la curva '
          'completa como resultado negativo (regla 10). Seguir con el paso 2.')
else:
    elegida = cands.sort_values('cobertura', ascending=False).iloc[0]
    print(f'✅ Cobertura elegida (la MAYOR que cumple): {elegida.cobertura:.2f}')
    print(f'   accuracy {elegida.accuracy:.4f} · trivial {elegida.trivial:.4f} · '
          f'brecha {elegida.brecha:+.4f} · balanced acc {elegida.balanced_acc:.4f} · '
          f'n = {int(elegida.n)}')
    print(f'   El sistema se abstendría en el {100*(1-elegida.cobertura):.0f} % de los casos.')
    print('\n⚠️  La cobertura quedó SELECCIONADA sobre OOF: este número es optimistamente '
          'sesgado. El no sesgado es el del cierre sobre Test, con la cobertura ya congelada.')
    print('⚠️  Verificar antes de festejar: ¿la brecha se mantiene o solo subió la accuracy '
          'porque el trivial del cubierto subió con ella? Está en la columna "brecha".')

json.dump({'brazo': BRAZO, 'curva': df.to_dict('records'),
           'candidatas': cands.to_dict('records')},
          open(f'{OUT_36}/paso1_selectiva.json', 'w'), indent=2, ensure_ascii=False)
print(f'\nGuardado en {OUT_36}/paso1_selectiva.json')
print('Evaluaciones sobre Test consumidas: 0.')

## 7. Qué sigue

1. Registrar la curva completa en `Fase3_Registro_de_Resultados.md`, cumpla o no el objetivo.
2. **La abstención también hay que implementarla en el producto** si se adopta: `detector.js`
   tendría que emitir un estado `sin_juicio` cuando `|p − 0,5|` cae bajo el umbral congelado,
   en vez de mostrar un porcentaje. Un número que el modelo no sostiene es peor que no mostrar
   nada. Anotarlo como requisito de implementación, no como detalle.
3. Pasar al **paso 2** (agregación con consecutividad corregida). La combinación es la
   interesante: las probabilidades promediadas por ventana están mejor separadas, así que la
   curva riesgo-cobertura sobre ventanas debería dominar a esta.